# EXAFS Optimizer
Plans the counting statistics of an EXAFS measurement **before** beamtime: how far in $k$ the signal of a given structure can be resolved above counting noise for a given count rate and counting time, and which $k$-weighted counting-time scheme reaches a target $k$ fastest.

The EXAFS signal $\chi(k)$ is simulated with FEFF (via [xraylarch](https://xraypy.github.io/xraylarch/)) from a CIF file and compared with the Poisson noise of the planned scan. The theory, advantages and limitations are described in `DOCUMENTATION.md`.

| Section | Question answered |
|---|---|
| 3. FEFF simulation | What signal does the structure give? |
| 4. Counting statistics | Noise for the chosen count rate and time per point → maximum resolvable $k$, $k_{res}$ |
| 5. $k_{res}$ vs counting time | How much time per point is needed for a given $k_{res}$? |
| 6. Effect on $R$ space | What does $\vert\chi(R)\vert$ look like with data up to $k_{res}$ only? |
| 7. $k$-weighted counting time | $k_{res}$ vs total time for $t(k) = t_0 (k/k_{start})^n$ |
| 8. Optimal exponent | Fastest $n$ for a target $k$, or best $k$ for a time budget |
| 9. Splitting into scans | How to split the plan of section 8 into scans, and what it costs in dead time |


A companion notebook, `EXAFS_MC_FIT.ipynb`, will estimate the parameter uncertainties of a full fit vs $k_{max}$ by Monte Carlo. It is *coming in a later release*.

**Workflow:** run **1. Setup** and **2. Control parameters**, then the sections. They are meant to be read in order, but each one computes what it needs from the earlier ones.

All computations and plots are in the `exafs_optimizer/` package next to this notebook (`planner.py` has one method per section). This notebook contains only the inputs and the outputs.

**Requirements:** Python ≥ 3.10, `numpy`, `scipy`, `pandas`, `matplotlib`, `xraylarch` (includes FEFF6/FEFF8 and pymatgen for CIF parsing), and the `exafs_optimizer/` package from this folder. Tested versions are pinned in `requirements.txt`. Example structures (public, from COD) are in `examples/`.

## 1. Setup
Imports the backend and sets the plot style. Run once.

In [ ]:
import numpy as np
from exafs_optimizer import Config, Planner, set_style, __version__
set_style()
print(f"EXAFS Optimizer {__version__}")

## 2. Control parameters
All user inputs. Edit here, re-run this cell, then re-run the sections you need. Paths can be absolute or relative to the notebook. Parameters left out take the defaults in `exafs_optimizer/config.py`.

In [ ]:
# ======================= CONTROL PARAMETERS =======================
cfg = Config(
    # ---- structure and FEFF simulation (section 3) ----
    cif_file      = "examples/UO2_COD9009049.cif",  # relative to the notebook, or an absolute path
    absorber      = "U",    # absorbing element symbol
    edge          = "L3",   # absorption edge
    absorber_site = 1,      # crystallographic site of absorber (if several)
    cluster_size  = 9.0,    # FEFF cluster radius (Å), must be > R_cut
    R_cut         = 7,      # keep paths with Reff < R_cut (Å)
    use_feff8     = True,   # True: feff8l, False: feff6l
    S02 = 1, E0 = 0.0, SIGMA2 = 0.005,            # path parameters applied to all paths
    kmin = 3.0, kmax = 15.0, kweight = 2,         # FT range and k-weight (kweight is used everywhere below)

    # ---- measurement (sections 4-8) ----
    count_rate = 1.6e3,     # count rate per data point (counts/s), background subtracted
    t_point    = 30,        # counting time per data point (s) -> N = count_rate * t_point counts
    edge_step  = 1.25,      # edge jump Delta(mu*x) of the absorber (transmission), or fraction of
                            # detected counts from the absorber edge (fluorescence); 1 = ideal
    E_start    = 30.0,      # start of the EXAFS scan above the edge (eV)
    K_scan_max = 13,        # end of the scan (Å^-1), at most the simulated grid end (20 Å^-1)
    dk_exp     = 0.05,      # k-spacing of the measured data (Å^-1)

    # ---- detection test (sections 4-8) ----
    win    = 1.5,           # analysis window width in k (Å^-1), ~1 oscillation of the main shell
    alpha  = 0.003,         # false-alarm probability (0.003 ~ 3 sigma)
    power  = 0.95,          # required probability of detecting the signal in a window

    # ---- k_res vs counting time per point (section 5) ----
    T_range    = np.logspace(-2, 4, 121),         # counting time per point (s)
    edge_steps = [1],                             # one curve per edge step

    # ---- R-space comparison and simulated scans (sections 6, 9) ----
    seed = 0,               # random seed of the noisy realisations

    # ---- k-weighted counting time (sections 7-8) ----
    time_exps = [0, 1, 2, 3],                     # time exponents n in t(k) = t0 (k/k_start)^n
    T0_range  = np.logspace(-3, 2, 101),          # t0: counting time per point at k_start (s)
    n_grid    = np.arange(0, 4.001, 0.01),        # exponents scanned by the optimisation
    k_target  = 12.0,       # (a) required k_res (Å^-1) -> n with the shortest total time
    T_budget  = 3.5,        # (b) total counting time available (h) -> n with the largest k_res
    t0_min    = "auto",     # minimum counting time per point at k_start (s), summed over all scans;
                            # "auto": the t0 that gives signal/noise = snr_lowk in the first window
    snr_lowk  = 5,          # required rms(chi)/sigma_chi per point in [k_start, k_start + win]

    # ---- splitting into scans (section 9) ----
    plan_from       = "a",  # "a": t0 and n of section 8(a) (k_target), "b": of section 8(b) (T_budget)
    margin          = 1.25, # safety factor on the counting time (1 = exactly at the detection limit)
    n_scans         = 20,   # number of scans the measurement is split into (averaged afterwards)
    t_dead_point    = 1,    # dead time per point in every scan (s): monochromator move, readout
    t_overhead_scan = 120,  # dead time per scan (s): scan start-up, file writing
)
plan = Planner(cfg)

## 3. FEFF simulation from CIF
CIF → `feff.inp` → FEFF run (in `<cif name>_feff/` next to the CIF) → sum of paths with $R_{eff}$ < `R_cut` → $\chi(k)$ and $|\chi(R)|$.

In [ ]:
plan.feff_simulation()

## 4. Counting statistics and maximum resolvable $k$
The count rate and the time per point give $N$ counts per point and the noise $\sigma_\chi = 1/(\sqrt{N}\,\Delta\mu x)$. In a sliding window of $M$ measured points the statistic $T=\sum\chi_{obs}^2/\sigma^2$ is $\chi^2_M$-distributed for pure noise and non-central $\chi^2_M(\lambda)$ with signal, $\lambda=\sum\chi^2/\sigma^2$. A window is resolved when $P(T > T_{thr}) \geq$ `power`, with $T_{thr}$ set by the false-alarm rate `alpha`. The largest resolved $k$ is $k_{res}$.

The scan grid runs from $E_0$ + `E_start` to `K_scan_max` in steps of `dk_exp`, and is used by sections 5–9.

In [ ]:
plan.counting_statistics()

## 5. Maximum resolvable $k$ vs counting time per point
$k_{res}$ for constant counting time per point, $N$ = `count_rate` × $t$. One curve per entry of `edge_steps`; the top axis gives the corresponding counts per point.

In [ ]:
plan.kres_vs_time()

## 6. Effect of the resolvable range in $R$ space
Fourier transform of $\chi(k)$ over the full range [`kmin`, `kmax`] (theory) compared with the range [`kmin`, $k_{res}$] that the measurement of section 4 can resolve, noiseless and for one noisy realisation. The shorter range lowers the $R$ resolution, $\delta R \approx \pi / (2\Delta k)$, broadening and merging the shells. $|\chi(R)|$ is not phase-corrected, so peaks sit ~0.3–0.5 Å below the true distances.

In [ ]:
plan.r_space()

## 7. $k$-weighted counting time: $k_{res}$ vs counting time
Counting time per point $t(k) = t_0\,(k/k_{start})^n$ for each $n$ in `time_exps`, so that $N(k)$ = `count_rate` $\cdot\,t(k)$ grows with $k$ and compensates the decay of $\chi(k)$.

In [ ]:
plan.kweighted_time()

## 8. Optimal time exponent $n$
Two complementary questions, scanned over the exponents in `n_grid`:

- **(a) target range:** the shortest total time that reaches $k_{res} \geq$ `k_target`. For each $n$, the smallest $t_0$ is found by bisection ($k_{res}$ never decreases with $t_0$).
- **(b) time budget:** the largest $k_{res}$ for a total counting time `T_budget`. For each $n$, $t_0$ = `T_budget` / $\sum_i (k_i/k_{start})^n$.

$k_{res}$ only tests the high-$k$ end, where the signal is weakest. Without a limit, both questions would push all the time there: the steeper $n$, the faster, so the optimum runs to the edge of `n_grid`. A fit, however, also needs good low-$k$ data (first shell, $E_0$, amplitude). `t0_min` is therefore a floor on $t_0$, the counting time per point at $k_{start}$ summed over all scans. With `"auto"` it follows from the simulated signal: the $t_0$ for which the noise per point equals the rms of $\chi$ in the first window [$k_{start}$, $k_{start}$ + `win`] divided by `snr_lowk`, i.e. $t_0 = \left(\mathrm{snr_{lowk}} / (\chi_{rms}\,\Delta\mu x)\right)^2 / R_0$. In (a), $t_0$ is raised to `t0_min` where the bisection gives less. In (b), exponents for which the budget gives $t_0$ < `t0_min` are excluded. Once $t_0$ reaches `t0_min`, a steeper $n$ only adds time at high $k$, so the optimum is an interior one.

Both use the scan grid and detection test of section 4. The time is pure counting time; a fixed dead time per point (monochromator moves, readout) adds the same constant for every $n$.

In [ ]:
plan.optimal_exponent()

## 9. Splitting the measurement into scans
Section 8 gives the counting scheme $t(k) = t_0\,(k/k_{start})^n$ as **one** measurement. In practice it is split into $N_s$ scans that are averaged. With counting noise only, $N_s$ scans with $t_0/N_s$ each collect the same counts per point as one scan with $t_0$. They therefore give the same noise and the same $k_{res}$. **The split does not change the statistics.** It only changes the dead time, which is paid in every scan, and the practicalities: checking the data as it comes in, discarding bad scans, drift, beam loss.

This section takes $t_0$ and $n$ from section 8(a) or 8(b) (`plan_from`) and multiplies the counting time by `margin`. Section 8 places $k_{res}$ exactly at the detection limit, and because $k_{res}$ jumps between groups of oscillations, a small shortfall can cost much more than a small amount of $k$. The counting time is then divided over `n_scans` scans.

The output lists the **instrument settings** for one scan: the scan range up to `K_scan_max`, $t(k)$ per scan and the number of scans. It also gives the total time including dead time (`t_dead_point` per point and `t_overhead_scan` per scan), and a table of the time for other numbers of scans. The counting time is the same for any number of scans; only the dead time grows.

The plot simulates the measurement: `n_scans` noisy scans with this $t(k)$ (random seed `seed`) and their average. (a) shows $k^w\chi(k)$ for one scan, the average and the noiseless signal. (b) shows $|\chi(R)|$ for each, all transformed over the resolved range [$k_{start}$, $k_{res}$].

In [ ]:
plan.split_scans()